In [1]:
import os
import pandas as pd
import numpy as np
from collections import defaultdict

In [2]:
PROJECT_ROOT = "/media/ausaafkhan/New Volume1/project_exhibition/broken_dataset_repair"

WORKING_PATH = os.path.join(PROJECT_ROOT, "data", "working", "df_working.csv")
PLAN_PATH = os.path.join(PROJECT_ROOT, "reports", "repair_plan", "repair_plan.csv")
OUTPUT_PATH = os.path.join(PROJECT_ROOT, "data", "cleaned", "df_cleaned.csv")

if not os.path.exists(WORKING_PATH):
    raise FileNotFoundError("df_working not found")

if not os.path.exists(PLAN_PATH):
    raise FileNotFoundError("repair_plan not found")

print("Using df_working:", WORKING_PATH)
print("Using repair_plan:", PLAN_PATH)

Using df_working: /media/ausaafkhan/New Volume1/project_exhibition/broken_dataset_repair/data/working/df_working.csv
Using repair_plan: /media/ausaafkhan/New Volume1/project_exhibition/broken_dataset_repair/reports/repair_plan/repair_plan.csv


In [3]:
df = pd.read_csv(WORKING_PATH)
repair_plan = pd.read_csv(PLAN_PATH)

# normalize column names
df.columns = df.columns.str.strip().str.lower()
repair_plan.columns = repair_plan.columns.str.strip().str.lower()

print("\nColumns:", df.columns.tolist())

print("\nMissing BEFORE repair:")
print(df.isna().sum().sort_values(ascending=False).head(10))


Columns: ['age', 'workclass', 'fnlwgt', 'education', 'educational-num', 'marital-status', 'occupation', 'relationship', 'race', 'gender', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']

Missing BEFORE repair:
occupation         2809
workclass          2799
native-country      857
fnlwgt                0
education             0
educational-num       0
age                   0
marital-status        0
relationship          0
gender                0
dtype: int64


In [4]:
def impute_missing(df, col):

    if col not in df.columns:
        return False

    if df[col].isna().sum() == 0:
        return False

    if pd.api.types.is_numeric_dtype(df[col]):
        value = df[col].median()
    else:
        mode_series = df[col].mode(dropna=True)
        if mode_series.empty:
            return False
        value = mode_series.iloc[0]

    before_na = df[col].isna().sum()
    df[col] = df[col].fillna(value)
    after_na = df[col].isna().sum()

    return after_na < before_na

In [5]:
total_actions = 0
action_counts = defaultdict(int)
columns_modified = set()

repair_plan = repair_plan.sort_values(
    by=["column_name", "issue_type"]
).reset_index(drop=True)

for _, row in repair_plan.iterrows():

    col = str(row["column_name"]).strip().lower()
    action = row["recommended_action"]
    strategy = row.get("strategy", None)

    # =========================
    # FAIL FAST (NO SILENT SKIP)
    # =========================
    if col not in df.columns:
        raise ValueError(f"Column missing in df: {col}")

    before = df[col].copy()
    changed = False

    # =========================
    # STRATEGY-BASED IMPUTATION
    # =========================
    if action == "impute_missing":

        if df[col].isna().sum() == 0:
            continue

        # 🔥 NUMERIC → MEDIAN
        if strategy == "median":
            numeric_col = pd.to_numeric(df[col], errors="coerce")
            value = numeric_col.median()

        # 🔥 CATEGORICAL → MODE
        elif strategy == "mode":
            mode_series = df[col].mode(dropna=True)
            if mode_series.empty:
                continue
            value = mode_series.iloc[0]

        else:
            raise ValueError(f"Missing or invalid strategy for column: {col}")

        before_na = df[col].isna().sum()

        df[col] = df[col].fillna(value)

        after_na = df[col].isna().sum()

        changed = after_na < before_na

    # =========================
    # OTHER ACTIONS
    # =========================
    elif action == "evaluate_high_cardinality":
        changed = False

    elif action == "mark_as_identifier":
        changed = False

    else:
        raise ValueError(f"Unsupported action: {action}")

    # =========================
    # TRACK CHANGES
    # =========================
    if changed:
        columns_modified.add(col)
        action_counts[action] += 1
        total_actions += 1

In [6]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

df.to_csv(OUTPUT_PATH, index=False)

print("\nSaved cleaned dataset to:", OUTPUT_PATH)

all_columns = set(df.columns)
columns_unchanged = all_columns - columns_modified

print("\n===== REPAIR EXECUTION AUDIT =====")
print(f"Total actions applied: {total_actions}")

print("\nActions per type:")
for k, v in sorted(action_counts.items()):
    print(f"{k}: {v}")

print("\nColumns modified:")
print(sorted(columns_modified))

print("\nColumns unchanged:")
print(sorted(columns_unchanged))


Saved cleaned dataset to: /media/ausaafkhan/New Volume1/project_exhibition/broken_dataset_repair/data/cleaned/df_cleaned.csv

===== REPAIR EXECUTION AUDIT =====
Total actions applied: 3

Actions per type:
impute_missing: 3

Columns modified:
['native-country', 'occupation', 'workclass']

Columns unchanged:
['age', 'capital-gain', 'capital-loss', 'education', 'educational-num', 'fnlwgt', 'gender', 'hours-per-week', 'income', 'marital-status', 'race', 'relationship']
